# Complete exploratory data analysis notebook

Author: Maria Jorda

Date: 22 Feb 2026

In this notebook we carry out a comprehensive exploratory data analysis for a dataset containing data of an E-commerce

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Data download

In [ ]:
# In order to pull the data you need to have the files in a folder in the same directory as this notebook. If you have the data in a different location, you can change the path in the code below.
relative_path="module_2_datasets/"

In [ ]:
df_orders = pd.read_csv(f'{relative_path}feature_frame.csv')
print(df_orders.shape)
df_orders.head()

## Sanity checks


### 1. Data types

In [ ]:
df_orders.dtypes

In [ ]:
df_orders.describe()

We should change the datatype of 'created_at' and 'order_date' because they're datetimes. Additionaly, I think it's better for boolean values to be intergers, not floats, so I'll also change the data types of boolean values. To know which variables are boolean we check the max and min of the variables (they should be 1 and 0 respectively) or the .value_counts() of the variables. I'll also change other float variables like 'count_adults' to interger.

In [ ]:
df_orders['created_at'] = df_orders['created_at'].astype('datetime64[ns]')
df_orders['order_date'] = df_orders['order_date'].astype('datetime64[ns]')

In [ ]:
print(df_orders['avg_days_to_buy_variant_id'].value_counts())

In [ ]:
boolean_columns = df_orders[['outcome', 'ordered_before', 'abandoned_before', 'active_snoozed', 'set_as_regular']]
interger_columns = df_orders[['days_since_purchase_variant_id', 'days_since_purchase_product_type', 'count_adults', 'count_children', 'count_babies', 'count_pets', 'people_ex_baby']]

for column in boolean_columns:
    df_orders[column] = df_orders[column].astype('int64')

for column in interger_columns:
    df_orders[column] = df_orders[column].astype('int64')

In [ ]:
df_orders.head()

In [ ]:
df_orders.dtypes

Now all the types are OK.

### 2. Missing values

In [ ]:
df_orders.isna().sum()

There are no missing values, this is surprising. I've run this same command just after downloading the data to double check that I've not deleted the NaNs when modifying datatypes, just in case, and there are no missing values.

## Data integrity checks

### 1. Distribution of numerical, non-boolean variables

In [ ]:
numerical_columns = df_orders[['user_order_seq', 'normalised_price', 'discount_pct', 'global_popularity', 'count_adults', 'count_children', 'count_babies', 'count_pets', 'people_ex_baby',
                                 'days_since_purchase_variant_id', 'avg_days_to_buy_variant_id', 'std_days_to_buy_variant_id', 'days_since_purchase_product_type', 'avg_days_to_buy_product_type', 'std_days_to_buy_product_type']]

fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(18, 20))
axes = axes.flatten()

for i,column in enumerate(numerical_columns):
    sns.histplot(df_orders[column], bins=30, ax=axes[i])
    axes[i].set_title(f'Distribution of {column}')
    axes[i].set_xlabel('Column')
    axes[i].set_ylabel('Frequency')
    
plt.tight_layout()
plt.show()

These charts are really valuable, since we infer some characteristics of the purchases. In the first graphs we see that the common user is the one that does little amount of purchases in this e-commerce (given that 'user_order_seq' is right-skewed) and that they tend to buy cheap products, although there are users with higher loyalty values, and expensive products too. Most of the customers are 2 adults per household without kids and pets, but there are also larger families. 
Days since last purchase (both of variant id and product type) is around 30 (one month after previous purchase) for many cases, which seems a bit weird unless there's maybe a special discount after one month from last purchase.
One thing that looks strange is that there are cases with discount_pct > 1, which would mean that the shop pays you, that makes no sense.

After reviewing all the distributions I consider that we only need to deal with the outliers of 'discount_pct', since I consider the other 'peaks' to be natural user behaviour (if this problem was real, I'd ask or look for data regarding the 30d re-purchase)

In [ ]:
df_orders[df_orders['discount_pct'] > 1]['discount_pct'].value_counts()

In [ ]:
# We will cap those values to 1, as it is not possible to have a discount higher than 100%
df_orders['discount_pct'] = df_orders['discount_pct'].apply(lambda x: 1 if x > 1 else x)

## Correlations between variables

In [ ]:
# Check the correlation between the numerical features with a heatmap
plt.figure(figsize=(12, 10))

id_columns = ['user_id', 'order_id', 'variant_id']
numerical_columns = df_orders.select_dtypes(include=['int64', 'float64'])
numerical_columns = numerical_columns.drop(columns=id_columns, errors='ignore')

corr_matrix = numerical_columns.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation heatmap')
plt.tight_layout()
plt.show()

We see a high correlation between 'people_ex_baby' and 'count_adults' and 'count_children', since people excluding babies is the sum of those variables, so 'people_ex_baby' should be deleted to avoid a multicollinearity problem. Then we also see a high correlation between 'avg_days_to_buy_variant_id' and 'std_days_to_buy_variant_id' which explains a pattern when users are buying: if they buy sth regularly (avg_days_to_buy_variant_id would be low) then the std_days_to_buy_variant_id is also low because there is little variation. Same logic for 'avg_days_to_buy_product_type' and 'std_days_to_buy_product_type'. For these variables I'm going to compute the VIF to see if there is a high multicollinearity. Values of VIF higher than 10 indicate a multicollinearity issue.

From the correlation matrix we also see that the variables that have more linear relation with 'outcome' (variable to predict) are 'ordered_before', 'days_since_purchase_product_type', 'active_snoozed', and the count_ variables.

In [ ]:
# VIF for numerical predictors to check for multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = numerical_columns.drop(columns=['outcome'], errors='ignore')
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

We see that the VIF is extremely high (inf) for count_adults, people_ex_baby and count_children, as people_ex_baby is explained by the other two. Additionally, days_since_purchase_variant_id, avg_days_to_buy_variant_id,std_days_to_buy_variant_id, avg_days_to_buy_product_type and std_days_to_buy_product_type also have VIF higher than 10, as days_since, avg and std are highly correlated. Keeping all variables could affect a predicting moddeling, so I'm going to drop the std variable (the avg variable is easier to understand)

In [ ]:
df_orders.drop(columns=['people_ex_baby', 'std_days_to_buy_variant_id', 'std_days_to_buy_product_type'], inplace=True)

In [ ]:
df_orders.dtypes

## Categorical encoding

We have one variable that has str values: 'product_type'. If we want to use it with a model, we should transform its values to a numerical format

In [ ]:
df_orders['product_type'].value_counts()

In [ ]:
df_orders['product_type'].unique()

We have many different types of product, so we cannot use one hot encoding, because that would create many many variables. Ordinal or label encoding would create an order that does not exist in the variable.
Using target encoding, a method that transforms categorical variables into numerical values based on the target variable, is more suitable in this scenario. Based on the documentation of the [TargetEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html) class, each category is encoded based on a shrunk estimate of the avg target values for observations belonging to the category. 

An important observation is that in order to mitigate overfitting, target encoding blends the global target mean (the prior, in Bayes terms) with the category-specific target mean (the posterior) using a regularization mechanism. This is achieved by using a smoothing parameter that ensures rare categories are not overly represented, thus reducing the risk of overfitting and improving the stability of the encoded values.

In [ ]:
# We use target encoding (from sklearn) for the product_type column, as it has a high cardinality and it is not ordinal
from sklearn.preprocessing import TargetEncoder

encoder = TargetEncoder(categories='auto', target_type='continuous', smooth='auto', cv=5, random_state=42)
df_orders['product_type_encoded'] = encoder.fit_transform(df_orders[['product_type']], df_orders['outcome'])

df_orders[['product_type', 'product_type_encoded', 'outcome']].sample(10)

If new values are inserted in 'product_type', this method does not fail, it applies the fallback: setting the avg probability of the entire catalog, since the new value would not have purchasing history, so the method is robust. If many new values are frequently created, we should consider retraining the TargetEncoding with some frequency (e.g., bimonthly)

EDA has been completed, although we could keep exploring the data as much as we want. Now we could proceed to a modelling step.